# ELHR + KG-RAG Evaluation — CSE499B
## Entity-Linked Hybrid Retrieval with Knowledge Graph Augmentation

### Three-channel fusion (Dense/Ch2 disabled — corpus/embedding misalignment):
### score(chunk) = α·RRF_BM25 + γ·RRF_MeSH + δ·RRF_KG

### Fixes applied vs original:
### 1. ch4_kg() — seed CUI selection uses CHD-count (not label length)
### 2. ch4_kg() — capped to 8 seed CUIs, forward-only traversal
### 3. fmt_passages() — source truncated at 2000 chars, retrieved at 800
### 4. get_question_cuis() Method 2 — IDF sort + word-boundary regex
### 5. extract_mesh_from_question() — word-boundary regex (no false matches)
### 6. Weights rebalanced — BM25=0.45, MeSH=0.30, KG=0.25
### 7. ALLOWED_KG_RELS — SY removed (zero new signal over PAR/CHD/RO/CO_OCCUR)
### 8. Dense channel reframed as excluded, not pending fix
### 9. K_KG_RRF lowered to 30 so KG rank-1 beats BM25 rank-1
### 10. External baselines added (Jin et al. 2019) to comparison table

## CELL 1: Install Dependencies

In [1]:
import subprocess, sys
for pkg in ['groq','rank_bm25','sentence-transformers','faiss-cpu','scikit-learn','tqdm','pyarrow','networkx']:
    r = subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'], capture_output=True, text=True)
    print(f'  {"✓" if r.returncode==0 else "✗"} {pkg}')
print('Done.')

  ✓ groq
  ✓ rank_bm25
  ✓ sentence-transformers
  ✓ faiss-cpu
  ✓ scikit-learn
  ✓ tqdm
  ✓ pyarrow
  ✓ networkx
Done.


## CELL 2: Imports and Groq API Keys

In [2]:
import gc, json, math, os, pickle, re, time, random, threading
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
from tqdm.auto import tqdm
from groq import Groq
import groq as groq_module
from rank_bm25 import BM25Okapi
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, cohen_kappa_score, confusion_matrix

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

GROQ_API_KEYS = []
for name in ['groq_API','groq_API_2','groq_API_3','groq_API_4','groq_API_5','groq_API_6',
             'groq_API_7','groq_API_8','groq_API_9','groq_API_10',
             'groq_API_11','groq_API_12','groq_API_13']:
    try:
        key = user_secrets.get_secret(name)
        if key and key.strip():
            GROQ_API_KEYS.append(key.strip())
            print(f'  ✓ {name}')
    except:
        print(f'  ✗ {name}')

print(f'\n{len(GROQ_API_KEYS)} API keys loaded')
assert len(GROQ_API_KEYS) >= 1

  ✓ groq_API
  ✓ groq_API_2
  ✓ groq_API_3
  ✓ groq_API_4
  ✓ groq_API_5
  ✓ groq_API_6
  ✓ groq_API_7
  ✓ groq_API_8
  ✓ groq_API_9
  ✓ groq_API_10
  ✓ groq_API_11
  ✓ groq_API_12
  ✓ groq_API_13

13 API keys loaded


## CELL 3: Configuration + Input Map
###   SET YOUR KAGGLE DATASET SLUG FOR hkg.pkl HERE

In [5]:
BASE_DIR  = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')

# INPUT 1 — slug only, no /kaggle/input prefix (avoids doubled path)
PQAL_DATASET_NAME   = 'datasets/konicarokeya/splits/splits'
FOLD0_DEV           = INPUT_DIR / PQAL_DATASET_NAME / 'pqal_fold0' / 'dev_set.json'

# INPUT 2
CORPUS_DATASET_NAME = 'datasets/konicarokeya/mesh-complete'
CORPUS_PARQUET      = INPUT_DIR / CORPUS_DATASET_NAME / 'retrieval_corpus_MESH_COMPLETE.parquet'

# INPUT 3
HKG_DATASET_NAME = '/kaggle/input/datasets/konicarokeya/hkg-umls'
HKG_PATH         = INPUT_DIR / HKG_DATASET_NAME / 'hkg.pkl'

# Outputs
CHECKPOINT_FILE      = BASE_DIR / 'fold0_checkpoint.json'
FINAL_RESULTS_FILE   = BASE_DIR / 'fold0_final_results.json'
THINKING_TRACES_FILE = BASE_DIR / 'fold0_thinking_traces.json'

# Model
MODEL_ID    = 'qwen/qwen3-32b'
TEMPERATURE = 0.6
MAX_TOKENS  = 1024

# ── Three-channel fusion weights (Dense/Ch2 disabled — alignment fix pending) ──
# Weight rationale (tuned on Fold 0 dev set, not seen during folds 2/4/6/8):
#   BM25=0.45  — strongest single channel; handles exact keyword queries well
#   MeSH=0.30  — entity-level precision boost for drug/disease terms
#   KG=0.25    — graph-expanded coverage; lower than BM25 due to traversal noise
#   Dense=0.00 — Channel 2 excluded: corpus chunk indices do not align with the
#                embedding index built in 499A. Reframed as three-channel system.
# K-constant rationale:
#   K_BM25_RRF=60 — standard RRF default; BM25 pool is large and diverse
#   K_MESH_RRF=40 — reduced so MeSH rank-1 outscores BM25 rank-32
#   K_KG_RRF=30   — lowered from 33: ensures KG rank-1 (0.00806) beats
#                   BM25 rank-1 (0.00738) so KG chunks win top-k slots
# TODO for paper: report Fold 0 grid search over
#   ALPHA in {0.40,0.45,0.50} x {0.25,0.30,0.35} x {0.20,0.25,0.30}
K_PASSAGES  = 15
ALPHA_BM25  = 0.45
ALPHA_DENSE = 0.00
ALPHA_MESH  = 0.30
ALPHA_KG    = 0.25

K_BM25_RRF = 60   # Channel 1
K_MESH_RRF = 40   # Channel 3 — MeSH rank-1 outscores BM25 rank-32
K_KG_RRF   = 30   # Channel 4 — lowered so KG rank-1 beats BM25 rank-1

# ── KG traversal settings ──────────────────────────────────────
KG_MAX_SEED_CUIS = 8
# SY (synonyms) removed — they map to the same concept, zero new signal
ALLOWED_KG_RELS  = {'PAR', 'CHD', 'RO', 'CO_OCCUR'}

# Rate limiting
TPM_LIMIT       = 300_000
RPM_LIMIT       = 1_000
MIN_REQUEST_GAP = 0.07

# Verify
print('='*65)
print('INPUT VERIFICATION')
print('='*65)
all_ok = True
for label, path in [
    ('INPUT 1 — dev_set.json (fold 0)',     FOLD0_DEV),
    ('INPUT 2 — corpus parquet (MeSH)',     CORPUS_PARQUET),
    ('INPUT 3 — hkg.pkl (knowledge graph)', HKG_PATH),
]:
    ok = Path(path).exists()
    if not ok: all_ok = False
    print(f'  {"✓" if ok else "✗ NOT FOUND"}  {label}')
    print(f'            → {path}')

print()
print('All inputs found ✓' if all_ok else '⚠  Fix missing paths above.')
print(f'\nWeights: BM25={ALPHA_BM25} | Dense={ALPHA_DENSE} (disabled) | MeSH={ALPHA_MESH} | KG={ALPHA_KG}')
print(f'Sum of weights: {ALPHA_BM25+ALPHA_DENSE+ALPHA_MESH+ALPHA_KG:.2f}  (should be 1.00)')
print(f'RRF constants: BM25 K={K_BM25_RRF} | MeSH K={K_MESH_RRF} | KG K={K_KG_RRF}')
print(f'Allowed KG rels: {ALLOWED_KG_RELS}')
print()
# Scoring sanity — verify KG rank-1 beats BM25 rank-1
tot = ALPHA_BM25 + ALPHA_MESH + ALPHA_KG
nb_ = ALPHA_BM25 / tot; nk_ = ALPHA_KG / tot
kg_r1   = nk_ / (K_KG_RRF   + 1)
bm25_r1 = nb_ / (K_BM25_RRF + 1)
print(f'KG   rank-1 score: {nk_:.4f}/({K_KG_RRF}+1) = {kg_r1:.5f}')
print(f'BM25 rank-1 score: {nb_:.4f}/({K_BM25_RRF}+1) = {bm25_r1:.5f}')
print(f'KG rank-1 > BM25 rank-1: {kg_r1 > bm25_r1}  ← must be True')


INPUT VERIFICATION
  ✓  INPUT 1 — dev_set.json (fold 0)
            → /kaggle/input/datasets/konicarokeya/splits/splits/pqal_fold0/dev_set.json
  ✓  INPUT 2 — corpus parquet (MeSH)
            → /kaggle/input/datasets/konicarokeya/mesh-complete/retrieval_corpus_MESH_COMPLETE.parquet
  ✓  INPUT 3 — hkg.pkl (knowledge graph)
            → /kaggle/input/datasets/konicarokeya/hkg-umls/hkg.pkl

All inputs found ✓

Weights: BM25=0.45 | Dense=0.0 (disabled) | MeSH=0.3 | KG=0.25
Sum of weights: 1.00  (should be 1.00)
RRF constants: BM25 K=60 | MeSH K=40 | KG K=30
Allowed KG rels: {'PAR', 'CHD', 'RO', 'CO_OCCUR'}

KG   rank-1 score: 0.2500/(30+1) = 0.00806
BM25 rank-1 score: 0.4500/(60+1) = 0.00738
KG rank-1 > BM25 rank-1: True  ← must be True


## CELL 4: Groq API Manager — 13-Key Rotation

In [6]:
class GroqAPIManager:
    def __init__(self, api_keys, model_id, tpm_limit=300_000, rpm_limit=1_000):
        self.model_id  = model_id
        self.tpm_limit = tpm_limit
        self.rpm_limit = rpm_limit
        self._lock     = threading.Lock()
        self.keys = [{'id':i,'key':k,'client':Groq(api_key=k),'tokens_used':0,
                      'requests_made':0,'window_start':time.time(),'failures':0,
                      'success_count':0,'rate_limited':False,'rate_limit_until':0.0}
                     for i,k in enumerate(api_keys)]
        self.current_key_idx = 0
        self.total_requests  = 0
        self.total_tokens    = 0
        self.failed_requests = []
        print(f'GroqAPIManager: {len(self.keys)} keys | {model_id}')

    def _reset_if_needed(self, ks):
        if time.time() - ks['window_start'] >= 60:
            ks.update({'tokens_used':0,'requests_made':0,'window_start':time.time(),
                       'rate_limited':False,'rate_limit_until':0.0})

    def _is_available(self, ks):
        self._reset_if_needed(ks)
        if ks['rate_limited'] and time.time() < ks['rate_limit_until']: return False
        if ks['tokens_used']   >= self.tpm_limit * 0.95: return False
        if ks['requests_made'] >= self.rpm_limit * 0.95: return False
        return True

    def _get_key(self):
        n = len(self.keys)
        for offset in range(n):
            idx = (self.current_key_idx + offset) % n
            if self._is_available(self.keys[idx]):
                self.current_key_idx = idx
                return self.keys[idx]
        now   = time.time()
        waits = [max(0, ks['rate_limit_until']-now) if ks['rate_limited']
                 else max(0, 60-(now-ks['window_start'])) for ks in self.keys]
        best  = waits.index(min(waits))
        print(f'   All keys exhausted. Waiting {waits[best]:.1f}s...')
        time.sleep(waits[best] + 1)
        ks = self.keys[best]
        ks.update({'tokens_used':0,'requests_made':0,'window_start':time.time(),
                   'rate_limited':False,'rate_limit_until':0.0})
        self.current_key_idx = best
        return ks

    def call(self, messages, temperature=0.6, max_tokens=1024, max_retries=5, question_id=None):
        last_error = None
        for attempt in range(max_retries):
            with self._lock:
                ks = self._get_key()
            try:
                time.sleep(MIN_REQUEST_GAP)
                resp = ks['client'].chat.completions.create(
                    model=self.model_id, messages=messages,
                    temperature=temperature, max_tokens=max_tokens, top_p=0.95)
                tokens = resp.usage.total_tokens if resp.usage else 500
                with self._lock:
                    ks['tokens_used']   += tokens
                    ks['requests_made'] += 1
                    ks['success_count'] += 1
                    self.total_requests += 1
                    self.total_tokens   += tokens
                    self.current_key_idx = (self.current_key_idx+1) % len(self.keys)
                full        = resp.choices[0].message.content or ''
                think_match = re.search(r'<think>(.*?)</think>', full, re.DOTALL)
                thinking    = think_match.group(1).strip() if think_match else ''
                final       = full[think_match.end():].strip() if think_match else full
                return final, thinking, tokens, ks['id']
            except groq_module.RateLimitError as e:
                last_error = e
                with self._lock:
                    ks['rate_limited']     = True
                    ks['rate_limit_until'] = time.time() + 60
                    ks['failures']        += 1
                print(f'   Key {ks["id"]} rate-limited → rotating (attempt {attempt+1}/{max_retries})')
            except Exception as e:
                last_error = e
                delay = 2.0 * (2**attempt) + random.uniform(0,1)
                print(f'  ⚠  Key {ks["id"]}: {type(e).__name__} → retry in {delay:.1f}s')
                with self._lock: ks['failures'] += 1
                time.sleep(delay)
        self.failed_requests.append({'question_id': question_id, 'error': str(last_error)})
        print(f'   FAILED after {max_retries} attempts for Q{question_id}')
        return None, None, 0, -1

    def status(self):
        print('\n' + '='*60)
        for ks in self.keys:
            s = 'limited' if ks['rate_limited'] and time.time()<ks['rate_limit_until'] else 'ok'
            print(f'  Key {ks["id"]:2d}: {s} OK={ks["success_count"]:4d} Fail={ks["failures"]:2d} Tokens={ks["tokens_used"]:,}')
        print(f'  Total: {self.total_requests} requests | {self.total_tokens:,} tokens | {len(self.failed_requests)} failed')
        print('='*60)


api_manager = GroqAPIManager(GROQ_API_KEYS, MODEL_ID, TPM_LIMIT, RPM_LIMIT)
print('API manager ready ✓')

GroqAPIManager: 13 keys | qwen/qwen3-32b
API manager ready ✓


## CELL 5: Load Fold 0 Dev Set (50 Questions)

In [7]:
print('='*65)
print('LOADING FOLD 0 DEV SET  [INPUT 1]')
print('='*65)

with open(FOLD0_DEV, 'r', encoding='utf-8') as f:
    raw_dev = json.load(f)

questions = []
for pubid, entry in raw_dev.items():
    contexts = entry.get('CONTEXTS', [])
    # No truncation here — full text kept in memory.
    # fmt_passages() in Cell 9 truncates at display time (2000/800 chars).
    # Truncating here AND there would silently cut the final sentence of
    # many abstracts, which usually contains the conclusion.
    source_ctx = ' '.join([c.strip() for c in contexts if c.strip()])
    questions.append({
        'pubid'          : str(pubid),
        'question'       : entry.get('QUESTION', ''),
        'source_context' : source_ctx,
        'contexts_list'  : contexts,
        'long_answer'    : entry.get('LONG_ANSWER', ''),
        'label'          : entry.get('final_decision', '').lower().strip(),
        'question_meshes': entry.get('MESHES', []),
    })

print(f'Loaded {len(questions)} questions')
dist = Counter(q['label'] for q in questions)
for lbl, cnt in sorted(dist.items()):
    print(f'  {lbl:8s}: {cnt} ({100*cnt/len(questions):.1f}%)')

# Show avg source length so we can verify truncation headroom
avg_len = sum(len(q['source_context']) for q in questions) / len(questions)
print(f'\n  Avg source_context length: {avg_len:.0f} chars')
print(f'  fmt_passages() will display first 2000 chars of source, 800 of retrieved')

LOADING FOLD 0 DEV SET  [INPUT 1]
Loaded 50 questions
  maybe   : 6 (12.0%)
  no      : 17 (34.0%)
  yes     : 27 (54.0%)

  Avg source_context length: 1325 chars
  fmt_passages() will display first 2000 chars of source, 800 of retrieved


## CELL 6A: Load Corpus — Text + Chunk IDs + MeSH Column

In [8]:
print('='*65)
print('LOADING CORPUS  [INPUT 2]')
print('='*65)

def normalize_mesh(m):
    if m is None: return []
    if isinstance(m, np.ndarray): m = m.tolist()
    if isinstance(m, list):
        result = []
        for x in m:
            if x is None: continue
            if isinstance(x, dict):
                term = x.get('term', '')
                if term and str(term).strip():
                    result.append(str(term).strip())
            else:
                s = str(x).strip()
                if s: result.append(s)
        return result
    try:
        if np.isnan(float(m)): return []
    except (TypeError, ValueError):
        pass
    return []

print('\n[1/2] Reading columns: text, chunk_id, meshes ...')
try:
    df = pd.read_parquet(CORPUS_PARQUET, columns=['text', 'chunk_id', 'meshes'])
    HAS_MESH_COL = True
    print(f'   {len(df):,} chunks loaded with meshes column')
except Exception as e:
    print(f'    Could not read meshes column: {e}')
    df = pd.read_parquet(CORPUS_PARQUET, columns=['text', 'chunk_id'])
    df['meshes'] = [[] for _ in range(len(df))]
    HAS_MESH_COL = False

corpus_texts = df['text'].tolist()
corpus_ids   = df['chunk_id'].tolist()

print('  Normalizing meshes...')
corpus_meshes = [normalize_mesh(m) for m in df['meshes'].tolist()]

del df
gc.collect()

n_with_mesh = sum(1 for m in corpus_meshes if len(m) > 0)
pct = 100 * n_with_mesh / len(corpus_meshes)
print(f'\n  MeSH coverage: {n_with_mesh:,}/{len(corpus_meshes):,} chunks ({pct:.1f}%)')
print(f'  Corpus size  : {len(corpus_texts):,} chunks')
print('\n[2/2] Corpus loaded.')

LOADING CORPUS  [INPUT 2]

[1/2] Reading columns: text, chunk_id, meshes ...
   2,089,296 chunks loaded with meshes column
  Normalizing meshes...

  MeSH coverage: 1,957,947/2,089,296 chunks (93.7%)
  Corpus size  : 2,089,296 chunks

[2/2] Corpus loaded.


## CELL 6B: Build BM25 Index — Channel 1

In [9]:
print('BUILDING BM25 INDEX  [Channel 1]')
print(f'Corpus size: {len(corpus_texts):,} chunks')

def tokenize(text):
    return str(text).lower().split()

def token_generator(texts):
    for t in tqdm(texts, desc='Tokenizing'):
        yield tokenize(t)

bm25_index = BM25Okapi(token_generator(corpus_texts))
gc.collect()
print(f'\nBM25 ready ✓  ({len(corpus_texts):,} docs indexed)')

BUILDING BM25 INDEX  [Channel 1]
Corpus size: 2,089,296 chunks


Tokenizing:   0%|          | 0/2089296 [00:00<?, ?it/s]


BM25 ready ✓  (2,089,296 docs indexed)


## CELL 6C: Build MeSH Inverted Index — Channel 3

In [10]:
print('='*65)
print('BUILDING MESH INVERTED INDEX  [Channel 3]')
print('='*65)

MESH_AVAILABLE      = False
mesh_inverted_index = {}
mesh_idf            = {}
mesh_vocabulary     = set()

if not HAS_MESH_COL:
    print('✗ meshes column missing — Channel 3 disabled.')
else:
    print('\n[1/3] Building inverted index...')
    raw_index = defaultdict(set)
    for idx, mesh_list in enumerate(tqdm(corpus_meshes, desc='Indexing MeSH')):
        for term in mesh_list:
            if term and str(term).strip():
                raw_index[str(term).strip().lower()].add(idx)

    print(f'  Unique MeSH terms: {len(raw_index):,}')

    print('\n[2/3] Computing IDF weights...')
    N = len(corpus_texts)
    for term, idxset in raw_index.items():
        df_cnt = len(idxset)
        mesh_idf[term] = math.log((N + 1) / (df_cnt + 1)) + 1

    print('\n[3/3] Finalizing...')
    mesh_inverted_index = {k: list(v) for k, v in raw_index.items()}
    mesh_vocabulary     = set(mesh_inverted_index.keys())
    del raw_index
    gc.collect()

    MESH_AVAILABLE = True
    print(f'\n✓ MeSH index ready | {len(mesh_inverted_index):,} terms | Channel 3 ACTIVE')

del corpus_meshes
gc.collect()
print('corpus_meshes freed ✓')

BUILDING MESH INVERTED INDEX  [Channel 3]

[1/3] Building inverted index...


Indexing MeSH:   0%|          | 0/2089296 [00:00<?, ?it/s]

  Unique MeSH terms: 26,190

[2/3] Computing IDF weights...

[3/3] Finalizing...

✓ MeSH index ready | 26,190 terms | Channel 3 ACTIVE
corpus_meshes freed ✓


## CELL 6D: Load Knowledge Graph — Channel 4 (NEW)
### Loads hkg.pkl built from UMLS via hkg-notebook.ipynb

In [ ]:
print('='*65)
print('LOADING KNOWLEDGE GRAPH  [Channel 4 — INPUT 3]')
print('Source: hkg.pkl (flat-dict format: adj, chunk_index, lookups)')
print('='*65)

KG_AVAILABLE     = False
G                = None   # rebuilt as nx.DiGraph from adj dict
label_to_cui     = {}
meshid_to_cui    = {}
synonym_to_cui   = {}
rxnorm_to_cui    = {}
norm_to_cui      = {}
cui_to_tui       = {}
cui_to_label     = {}
term_to_chunks   = {}
meshid_to_chunks = {}

if not HKG_PATH.exists():
    print(f'✗ Not found: {HKG_PATH}')
    print('  Set HKG_DATASET_NAME correctly in Cell 3.')
else:
    try:
        print(f'[1/4] Loading hkg.pkl ({HKG_PATH.stat().st_size/1024**2:.0f} MB)...')
        with open(HKG_PATH, 'rb') as f:
            hkg = pickle.load(f)

        # ── New flat-dict format keys ──
        adj              = hkg['adj']            # {cui: [(nbr, rel, rela, weight), ...]}
        chunk_index      = hkg['chunk_index']    # {cui: [chunk_idx, ...]}
        cui_to_tui       = hkg['cui_to_tui']
        cui_to_label     = hkg['cui_to_label']
        label_to_cui     = hkg['label_to_cui']
        meshid_to_cui    = hkg['meshid_to_cui']
        synonym_to_cui   = hkg['synonym_to_cui']
        rxnorm_to_cui    = hkg['rxnorm_to_cui']
        norm_to_cui      = hkg['norm_to_cui']
        term_to_chunks   = hkg.get('term_to_chunks',   {})
        meshid_to_chunks = hkg.get('meshid_to_chunks', {})
        del hkg
        gc.collect()

        print(f'[2/4] Rebuilding nx.DiGraph from adj dict ({len(adj):,} nodes)...')
        G = nx.DiGraph()

        # Add all nodes (including those with no edges but with chunks)
        all_cuis = set(adj.keys()) | set(chunk_index.keys())
        for cui in tqdm(all_cuis, desc='Adding nodes'):
            G.add_node(cui,
                       label=cui_to_label.get(cui, ''),
                       chunk_idxs=chunk_index.get(cui, []))

        # Add edges from adj
        for cui, nbrs in tqdm(adj.items(), desc='Adding edges'):
            for (nbr, rel, rela, weight) in nbrs:
                if nbr not in G:
                    G.add_node(nbr,
                               label=cui_to_label.get(nbr, ''),
                               chunk_idxs=chunk_index.get(nbr, []))
                G.add_edge(cui, nbr, rel=rel, rela=rela, weight=weight)

        del adj, chunk_index
        gc.collect()

        print(f'[3/4] Verifying graph...')
        n_nodes = G.number_of_nodes()
        n_edges = G.number_of_edges()
        n_with  = sum(1 for n in G.nodes() if G.nodes[n].get('chunk_idxs'))

        print(f'  Nodes              : {n_nodes:,}')
        print(f'  Edges              : {n_edges:,}')
        print(f'  Nodes w/ chunks    : {n_with:,} ({100*n_with/max(n_nodes,1):.1f}%)')
        print(f'  label_to_cui       : {len(label_to_cui):,} terms')
        print(f'  meshid_to_cui      : {len(meshid_to_cui):,} D-numbers')
        print(f'  synonym_to_cui     : {len(synonym_to_cui):,}')
        print(f'  rxnorm_to_cui      : {len(rxnorm_to_cui):,}')
        print(f'  norm_to_cui        : {len(norm_to_cui):,}')

        print(f'\n[4/4] Traversal check:')
        test_terms = ['diabetes mellitus', 'metformin', 'hypertension', 'insulin']
        for term in test_terms:
            cui = label_to_cui.get(term)
            if not cui:
                print(f'    {term:30s} → NOT IN GRAPH')
                continue
            chunks = G.nodes[cui].get('chunk_idxs', [])
            nbrs   = [G.nodes[n].get('label', '?') for n in list(G.successors(cui))[:2]]
            print(f'    {term:30s} → {len(chunks):,} chunks | neighbours: {nbrs}')

        KG_AVAILABLE = True
        print(f'\n✓ KG loaded | Channel 4 ACTIVE')

    except Exception as e:
        import traceback
        print(f'✗ Failed to load KG: {type(e).__name__}: {e}')
        traceback.print_exc()
        print('  Channel 4 disabled.')

print(f'\nChannel status:')
print(f'  Ch1 BM25         : ✓ Active')
print(f'  Ch2 Dense FAISS  : ✗ Disabled')
print(f'  Ch3 MeSH index   : {"✓ Active" if MESH_AVAILABLE else "✗ Disabled"}')
print(f'  Ch4 KG traversal : {"✓ Active" if KG_AVAILABLE else "✗ Disabled"}')

LOADING KNOWLEDGE GRAPH  [Channel 4 — INPUT 3]
Source: hkg.pkl built from UMLS (MSH + SNOMEDCT_US)
[1/2] Loading hkg.pkl (669 MB)...
[2/2] Verifying graph...
  Nodes          : 852,797
  Edges          : 3,734,427
  Nodes w/ chunks: 32,820 (3.8%)
  label_to_cui   : 2,419,044 terms
  meshid_to_cui  : 28,602 D-numbers

  Traversal check:
    diabetes mellitus              → 9,262 chunks | neighbours: ['aspergillus fumigates', 'disorder of endocrine pancreas (disorder)']
    metformin                      → 1,278 chunks | neighbours: ['metformin hydrochloride (substance)', 'sitagliptin phosphate-metformin hydrochloride drug combination']
    hypertension                   → 30,590 chunks | neighbours: ['hypertension nos (& [essential]) (disorder)', 'acebutolol (substance)']
    insulin                        → 25,464 chunks | neighbours: ['cisapride (substance)', 'gastrointestinal hormone (substance)']

✓ KG loaded | Channel 4 ACTIVE

Channel status:
  Ch1 BM25         : ✓ Active
  Ch2 De

## CELL 7: MeSH + CUI Extractor for Questions

Sample Q: Is cytokeratin immunoreactivity useful in the diagnosis of short-segment Barrett...
MeSH terms (18): ['adult', 'aged', 'barrett esophagus', 'biomarkers', 'biopsy']
KG CUIs   (18): ['C0036668', 'C0672250', 'C0001792', 'C0021769', 'C0086582']
  C0036668: "sensitivity specificity" | 19935 chunks | 2 successors
  C0672250: "keratin ck7" | 106 chunks | 24 successors
  C0001792: "elderly person (person)" | 416627 chunks | 11606 successors
  C0021769: "fibroblast intermediate filament proteins" | 1250 chunks | 285 successors
  C0086582: "male structure" | 931964 chunks | 17119 successors
Extractors ready ✓


## CELL 8: Four-Channel Retrieval Functions

In [13]:
K_RRF     = K_BM25_RRF   # backward compat alias
POOL_SIZE = K_PASSAGES * 4


# ── Channel 1: BM25 ──────────────────────────────────────────
def ch1_bm25(query_text, pool):
    scores   = bm25_index.get_scores(tokenize(query_text))
    top_idxs = np.argsort(scores)[::-1][:pool]
    return {int(i): r+1 for r, i in enumerate(top_idxs) if scores[i] > 0}


# ── Channel 2: Dense FAISS (disabled — corpus/embedding index misalignment) ──
def ch2_dense(query_text, pool):
    return {}


# ── Channel 3: MeSH Inverted Index ───────────────────────────
def ch3_mesh(query_mesh_terms, pool):
    if not MESH_AVAILABLE or not query_mesh_terms:
        return {}
    chunk_scores = defaultdict(float)
    for term in query_mesh_terms:
        norm = term.strip().lower()
        if norm not in mesh_inverted_index:
            continue
        idf = mesh_idf.get(norm, 1.0)
        for cidx in mesh_inverted_index[norm]:
            chunk_scores[cidx] += idf
    if not chunk_scores:
        return {}
    ranked = sorted(chunk_scores.items(), key=lambda x: x[1], reverse=True)[:pool]
    return {int(idx): r+1 for r, (idx, _) in enumerate(ranked)}


# ── Channel 4: KG Graph Traversal ────────────────────────────
def ch4_kg(question_cuis, pool):
    """
    Forward-only traversal from seed CUIs.
    Seed CUIs capped at KG_MAX_SEED_CUIS=8, selected by UMLS hierarchy
    specificity: fewest CHD (child) edges = most specific leaf node.
    Label length is NOT used — short labels are not more specific
    (e.g. 'TB' is shorter than 'pulmonary tuberculosis' but less specific).
    Direct CUI chunks score 1.0, neighbour chunks score 0.5.
    """
    if not KG_AVAILABLE or not question_cuis:
        return {}

    # Cap to most specific CUIs — fewest CHD children = most specific leaf node.
    # Leaf concepts in UMLS have 0-2 children; broad categories have dozens.
    sorted_cuis = sorted(
        question_cuis,
        key=lambda c: sum(1 for nbr in G.successors(c)
                          if G.edges[c, nbr].get('rel') == 'CHD')
                      if c in G else 999
    )[:KG_MAX_SEED_CUIS]

    direct_cuis   = set(sorted_cuis)
    expanded_cuis = set(sorted_cuis)

    for cui in list(direct_cuis):
        if cui not in G:
            continue
        for nbr in G.successors(cui):
            if G.edges[cui, nbr].get('rel') in ALLOWED_KG_RELS:
                expanded_cuis.add(nbr)

    chunk_score = defaultdict(float)
    for cui in expanded_cuis:
        if cui not in G:
            continue
        chunks = G.nodes[cui].get('chunk_idxs', [])
        weight = 1.0 if cui in direct_cuis else 0.5
        for idx in chunks:
            if 0 <= idx < len(corpus_texts):
                chunk_score[idx] += weight

    if not chunk_score:
        return {}

    ranked = sorted(chunk_score.items(), key=lambda x: -x[1])[:pool]
    return {int(idx): rank + 1 for rank, (idx, _) in enumerate(ranked)}


# ── Weighted RRF Fusion ───────────────────────────────────────
def elhr_kg_retrieve(q_dict, k):
    """
    Three-channel RRF with split K constants (Dense/Ch2 disabled).

    Why split K matters (weights normalised: BM25=0.45, MeSH=0.30, KG=0.25):
      K_BM25_RRF=60:  BM25 rank-1 score = 0.45/61 = 0.00738
      K_MESH_RRF=40:  MeSH rank-1 score = 0.30/41 = 0.00732
      K_KG_RRF=30:    KG   rank-1 score = 0.25/31 = 0.00806  ← beats BM25

    Before fix (single K=60):
      KG rank-1 score    = 0.25/61 = 0.00410
      BM25 rank-60 score = 0.45/120= 0.00375
      → KG rank-1 barely beat BM25 rank-60, KG chunks never won slots.

    After fix (K_KG_RRF=30):
      KG rank-1 = 0.00806 > BM25 rank-1 = 0.00738
      → KG-exclusive chunks now displace BM25 results in top-k.
    """
    query_text       = q_dict['question']
    query_mesh_terms = extract_mesh_from_question(q_dict)
    question_cuis    = get_question_cuis(q_dict)

    r_bm25  = ch1_bm25(query_text, POOL_SIZE)
    r_dense = ch2_dense(query_text, POOL_SIZE)
    r_mesh  = ch3_mesh(query_mesh_terms, POOL_SIZE)
    r_kg    = ch4_kg(question_cuis, POOL_SIZE)

    # Renormalise weights for active channels only
    a_b = ALPHA_BM25  if r_bm25  else 0.0
    a_d = ALPHA_DENSE if r_dense else 0.0
    a_m = ALPHA_MESH  if r_mesh  else 0.0
    a_k = ALPHA_KG    if r_kg    else 0.0
    total = (a_b + a_d + a_m + a_k) or 1.0
    nb, nd, nm, nk = a_b/total, a_d/total, a_m/total, a_k/total

    all_cands = set(r_bm25) | set(r_dense) | set(r_mesh) | set(r_kg)
    fused = []
    for idx in all_cands:
        score = (
            nb / (K_BM25_RRF + r_bm25.get(idx,  POOL_SIZE+1)) +
            nd / (K_BM25_RRF + r_dense.get(idx, POOL_SIZE+1)) +
            nm / (K_MESH_RRF + r_mesh.get(idx,  POOL_SIZE+1)) +
            nk / (K_KG_RRF   + r_kg.get(idx,    POOL_SIZE+1))
        )
        fused.append((
            idx, score,
            {
                'in_bm25'  : idx in r_bm25,
                'in_dense' : idx in r_dense,
                'in_mesh'  : idx in r_mesh,
                'in_kg'    : idx in r_kg,
                'bm25_rank': r_bm25.get(idx),
                'mesh_rank': r_mesh.get(idx),
                'kg_rank'  : r_kg.get(idx),
                'is_source': False,
                'n_kg_cuis': len(question_cuis),
            }
        ))

    fused.sort(key=lambda x: x[1], reverse=True)
    passages = [
        (corpus_texts[i], score, bd)
        for i, score, bd in fused[:k]
        if 0 <= i < len(corpus_texts)
    ]
    return passages, query_mesh_terms


def elhr_kg_retrieve_with_source(q_dict, k):
    """
    PubMedQA standard protocol:
      [0]     = Source Abstract (from dev_set.json CONTEXTS) — always correct
      [1..k-1]= ELHR+KG corpus retrieval — corroborating evidence
    """
    source_text    = q_dict.get('source_context', '').strip()
    source_passage = (
        source_text, 1.0,
        {'in_bm25': False, 'in_dense': False, 'in_mesh': False, 'in_kg': False,
         'bm25_rank': None, 'mesh_rank': None, 'kg_rank': None,
         'is_source': True, 'n_kg_cuis': 0}
    )
    retrieved, mesh_terms = elhr_kg_retrieve(q_dict, k=k-1)
    return [source_passage] + retrieved, mesh_terms


# ── Sanity test ───────────────────────────────────────────────
if questions:
    q0        = questions[0]
    q0_cuis   = get_question_cuis(q0)
    r_kg_test = ch4_kg(q0_cuis, POOL_SIZE)
    r_b_test  = ch1_bm25(q0['question'], POOL_SIZE)

    a_b = ALPHA_BM25; a_m = ALPHA_MESH; a_k = ALPHA_KG
    tot = a_b + a_m + a_k
    nb_  = a_b / tot
    nk_  = a_k / tot

    kg_rank1_score   = nk_ / (K_KG_RRF   + 1)
    bm25_rank1_score = nb_ / (K_BM25_RRF + 1)
    bm25_rank60_score= nb_ / (K_BM25_RRF + 60)

    print('── Scoring sanity check ─────────────────────────────')
    print(f'  KG   rank-1  score : {nk_:.4f} / ({K_KG_RRF}+1)   = {kg_rank1_score:.5f}')
    print(f'  BM25 rank-1  score : {nb_:.4f} / ({K_BM25_RRF}+1)  = {bm25_rank1_score:.5f}')
    print(f'  BM25 rank-60 score : {nb_:.4f} / ({K_BM25_RRF}+60) = {bm25_rank60_score:.5f}')
    print(f'  KG rank-1 > BM25 rank-1  : {kg_rank1_score > bm25_rank1_score}  ← must be True')
    print(f'  KG pool candidates: {len(r_kg_test)}')
    print()

    print('── Full retrieval test ──────────────────────────────')
    t_pass, t_mesh = elhr_kg_retrieve_with_source(q0, k=6)
    print(f'  Q : {q0["question"][:70]}...')
    print(f'  Label : {q0["label"]}')
    print(f'  MeSH  : {t_mesh[:4]}')
    print(f'  CUIs  : {q0_cuis[:4]}')
    print(f'\n  Top-6 passages (1 source + 5 retrieved):')
    kg_wins = 0
    for i, (txt, score, bd) in enumerate(t_pass):
        if bd.get('is_source'):
            tag = 'SOURCE'
        else:
            ch = '+'.join([c for c,v in [('BM25',bd['in_bm25']),
                           ('MeSH',bd['in_mesh']),('KG',bd['in_kg'])] if v])
            tag = f'{ch}'
            if bd['in_kg'] and not bd['in_bm25'] and not bd['in_mesh']:
                kg_wins += 1
        print(f'  [{i+1}] {tag:20s} score={score:.5f}  |  {txt[:60]}...')
    print(f'\n  KG-exclusive passages in top-6: {kg_wins}  ← should be > 0 after fix')

print('\nRetrieval ready ✓')


── Scoring sanity check ─────────────────────────────
  KG   rank-1  score : 0.2500 / (30+1)   = 0.00806
  BM25 rank-1  score : 0.4500 / (60+1)  = 0.00738
  BM25 rank-60 score : 0.4500 / (60+60) = 0.00375
  KG rank-1 > BM25 rank-1  : True  ← must be True
  KG pool candidates: 60

── Full retrieval test ──────────────────────────────
  Q : Is cytokeratin immunoreactivity useful in the diagnosis of short-segme...
  Label : yes
  MeSH  : ['adult', 'aged', 'barrett esophagus', 'biomarkers']
  CUIs  : ['C0036668', 'C0672250', 'C0001792', 'C0021769']

  Top-6 passages (1 source + 5 retrieved):
  [1] SOURCE               score=1.00000  |  Cytokeratin 7/20 staining has been reported to be helpful in...
  [2] KG                   score=0.01475  |  Recent advances in analysis of leukemic cell phenotypes usin...
  [3] KG                   score=0.01450  |  The clinical significance of immunodiagnosis of leukemia cel...
  [4] KG                   score=0.01427  |  Primary tissue culture methods ha

## CELL 9: Prompt Templates

In [14]:
SYS = """You are an expert biomedical researcher. Answer ONLY based on the retrieved evidence.
Output exactly one word: yes, no, or maybe.
Rules:
- yes   = evidence clearly supports a positive answer
- no    = evidence clearly contradicts the question
- maybe = evidence is mixed, contradictory, insufficient, or genuinely inconclusive
IMPORTANT: 'maybe' is rare. Reserve it only when evidence is directly contradictory
or completely silent on the question. Do not use maybe simply because the topic is complex.
End with: ANSWER: [yes/no/maybe]"""


def fmt_passages(passages):
    """
    Truncate at display time only (single truncation point).
    Source abstract: 2000 chars — enough to keep the conclusion sentence.
    Retrieved passages: 800 chars — sufficient context without token bloat.
    """
    parts = []
    for i, (txt, score, bd) in enumerate(passages):
        if bd.get('is_source'):
            header = f'[Passage {i+1} | Source Abstract — the study this question is about]'
            body   = txt.strip()[:2000]   # raised from 1200 — keeps conclusion
        else:
            chs = '+'.join([c for c, v in [
                ('BM25', bd['in_bm25']),
                ('MeSH', bd['in_mesh']),
                ('KG',   bd['in_kg']),
            ] if v])
            header = f'[Passage {i+1} | Retrieved via: {chs}]'
            body   = txt.strip()[:800]
        parts.append(f'{header}\n{body}')
    return '\n\n'.join(parts)


def prompt_basic(q, passages, mesh_terms):
    ctx  = fmt_passages(passages)
    user = (
        f'RETRIEVED EVIDENCE:\n{ctx}\n\n'
        f'QUESTION: {q}\n\n'
        f'Note: Passage 1 is the source abstract directly related to this question.\n'
        f'Use maybe only if evidence is directly contradictory or completely absent.\nANSWER:'
    )
    return [{'role': 'system', 'content': SYS}, {'role': 'user', 'content': user}]


def prompt_cot(q, passages, mesh_terms):
    ctx  = fmt_passages(passages)
    user = (
        f'RETRIEVED EVIDENCE:\n{ctx}\n\n'
        f'QUESTION: {q}\n\n'
        f'Note: Passage 1 is the source abstract directly related to this question.\n\n'
        f'Reason step by step:\n'
        f'1. What does the source abstract (Passage 1) conclude?\n'
        f'2. Do the KG-retrieved passages support or contradict it?\n'
        f'3. Is the overall evidence consistent, contradictory, or completely absent?\n'
        f'4. Use maybe only for genuine contradiction or total absence of evidence.\nANSWER:'
    )
    return [{'role': 'system', 'content': SYS}, {'role': 'user', 'content': user}]


def prompt_evidence(q, passages, mesh_terms):
    ctx      = fmt_passages(passages)
    mesh_str = ', '.join(mesh_terms[:8]) if mesh_terms else 'none'
    user = (
        f'Key biomedical concepts (MeSH + KG): {mesh_str}\n\n'
        f'RETRIEVED EVIDENCE:\n{ctx}\n\n'
        f'QUESTION: {q}\n\n'
        f'Note: Passage 1 is the source abstract. KG-retrieved passages provide related evidence.\n\n'
        f'- yes  ONLY if majority evidence consistently supports the question\n'
        f'- no   ONLY if majority evidence clearly contradicts it\n'
        f'- maybe ONLY if evidence is directly contradictory or completely absent\nANSWER:'
    )
    return [{'role': 'system', 'content': SYS}, {'role': 'user', 'content': user}]


def prompt_fewshot(q, passages, mesh_terms):
    ctx = fmt_passages(passages)
    ex  = (
        'Ex1: Source abstract shows aspirin consistently reduces CV events. ANSWER: yes\n'
        'Ex2: Source abstract shows no significant association found. ANSWER: no\n'
        'Ex3: Two studies directly contradict each other with no consensus. ANSWER: maybe\n\n'
        'Note: maybe requires direct contradiction or total evidence absence — not just complexity.\n\n'
    )
    user = (
        f'{ex}Now answer:\n\n'
        f'RETRIEVED EVIDENCE:\n{ctx}\n\n'
        f'QUESTION: {q}\n'
        f'Note: Passage 1 is the source abstract. Additional passages are KG/MeSH retrieved.\nANSWER:'
    )
    return [{'role': 'system', 'content': SYS}, {'role': 'user', 'content': user}]


PROMPT_BUILDERS      = {
    'basic'   : prompt_basic,
    'cot'     : prompt_cot,
    'evidence': prompt_evidence,
    'fewshot' : prompt_fewshot,
}
PROMPT_TYPES_TO_TEST = ['basic', 'cot', 'evidence', 'fewshot']
print('Prompts ready:', list(PROMPT_BUILDERS.keys()))

Prompts ready: ['basic', 'cot', 'evidence', 'fewshot']


## CELL 10: Answer Parser + Metrics

In [15]:
def parse_answer(text):
    if not text: return 'invalid'
    t = text.strip().lower()
    m = re.search(r'answer\s*:\s*(yes|no|maybe)', t)
    if m: return m.group(1)
    lines = [l.strip() for l in t.split('\n') if l.strip()]
    if lines:
        for lbl in ['maybe','yes','no']:
            if re.search(rf'\b{lbl}\b', lines[-1]): return lbl
    if re.search(r'\bmaybe\b', t): return 'maybe'
    if re.search(r'\byes\b',   t): return 'yes'
    if re.search(r'\bno\b',    t): return 'no'
    return 'invalid'


def compute_metrics(y_true, y_pred):
    labels = ['yes','no','maybe']
    valid  = [(t,p) for t,p in zip(y_true,y_pred) if p!='invalid']
    n_inv  = len(y_true) - len(valid)
    if not valid: return {'error':'no valid predictions'}
    yt, yp = [v[0] for v in valid], [v[1] for v in valid]
    m = {'n_total':len(y_true),'n_valid':len(valid),'n_invalid':n_inv,
         'accuracy':round(accuracy_score(yt,yp)*100,2),
         'f1_macro':round(f1_score(yt,yp,labels=labels,average='macro',zero_division=0)*100,2),
         'kappa':round(cohen_kappa_score(yt,yp,labels=labels),4),
         'precision':round(precision_score(yt,yp,labels=labels,average='macro',zero_division=0)*100,2),
         'recall':round(recall_score(yt,yp,labels=labels,average='macro',zero_division=0)*100,2),
         'pred_dist':dict(Counter(yp)),
         'conf_matrix':confusion_matrix(yt,yp,labels=labels).tolist()}
    for lbl in labels:
        yb=[1 if x==lbl else 0 for x in yt]; pb=[1 if x==lbl else 0 for x in yp]
        tp=sum(1 for a,b in zip(yb,pb) if a==b==1)
        fp=sum(1 for a,b in zip(yb,pb) if a==0 and b==1)
        fn=sum(1 for a,b in zip(yb,pb) if a==1 and b==0)
        p=tp/(tp+fp) if tp+fp>0 else 0
        r=tp/(tp+fn) if tp+fn>0 else 0
        f=2*p*r/(p+r) if p+r>0 else 0
        m[f'p_{lbl}']=round(p*100,2); m[f'r_{lbl}']=round(r*100,2); m[f'f_{lbl}']=round(f*100,2)
    return m


def print_metrics(m, title=''):
    print(f'\n{"-"*65}')
    print(f'  {title}')
    print(f'{"-"*65}')
    print(f'  Accuracy  : {m["accuracy"]}%')
    print(f'    vs Random baseline (3-class)        : 33.3%')
    print(f'    vs Jin et al. 2019 SVM (PubMedQA)   : 55.8%')
    print(f'    vs Jin et al. 2019 BioBERT fine-tune : 68.1%  ← published upper bound')
    print(f'    vs 499A CV baseline (BM25+Dense)     : 52.8%')
    print(f'  F1 Macro  : {m["f1_macro"]}%   (499A baseline: 34.1%)')
    print(f'  Kappa     : {m["kappa"]}   (499A baseline: 0.074)')
    print(f'  Precision : {m["precision"]}%')
    print(f'  Recall    : {m["recall"]}%')
    print(f'  Valid/Total: {m["n_valid"]}/{m["n_total"]}  invalid={m["n_invalid"]}')
    print(f'  Per-class:')
    for lbl in ['yes','no','maybe']:
        print(f'    {lbl:6s}: P={m[f"p_{lbl}"]:5.1f}%  R={m[f"r_{lbl}"]:5.1f}%  F1={m[f"f_{lbl}"]:5.1f}%')
    print(f'  Preds: {m["pred_dist"]}')
    # Confusion matrix — most diagnostic output for 3-class imbalanced problem
    print(f'  Confusion matrix (rows=true, cols=pred) [yes, no, maybe]:')
    for row in m['conf_matrix']:
        print(f'    {row}')


print('Parser + metrics ready ✓')


Parser + metrics ready ✓


## CELL 11: Checkpoint Helpers

In [16]:
def load_ckpt(path):
    if Path(path).exists():
        with open(path,'r') as f: data = json.load(f)
        print(f'Checkpoint: {len(data)} questions done')
        return data
    print('No checkpoint — fresh start')
    return {}

def save_ckpt(path, data):
    with open(path,'w') as f: json.dump(data, f, indent=2, ensure_ascii=False)

print('Checkpoint helpers ready ✓')

Checkpoint helpers ready ✓


## CELL 11B: Reusable Fold Evaluation Function
### This single function handles ALL folds — no code duplication

In [17]:
def run_fold(fold_num, dev_path, ckpt_path, results_path, traces_path,
             prompt_types, best_prompt=None):
    run_prompts = [best_prompt] if best_prompt else prompt_types

    print('='*65)
    print(f'FOLD {fold_num} — ELHR+KG EVALUATION')
    print('='*65)
    print(f'  Prompts  : {run_prompts}')
    print(f'  Channels : BM25(α={ALPHA_BM25}) | MeSH(γ={ALPHA_MESH}) | KG(δ={ALPHA_KG})')
    print(f'  KG active: {KG_AVAILABLE}  |  MeSH active: {MESH_AVAILABLE}')
    print(f'  Model    : {MODEL_ID}  |  k={K_PASSAGES}')

    if not Path(dev_path).exists():
        raise FileNotFoundError(f'Fold {fold_num} dev_set.json not found: {dev_path}')
    with open(dev_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)

    fold_qs = []
    for pubid, entry in raw.items():
        contexts = entry.get('CONTEXTS', [])
        source_ctx = ' '.join([c.strip() for c in contexts if c.strip()])
        fold_qs.append({
            'pubid'          : str(pubid),
            'question'       : entry.get('QUESTION', ''),
            'source_context' : source_ctx,
            'contexts_list'  : contexts,
            'long_answer'    : entry.get('LONG_ANSWER', ''),
            'label'          : entry.get('final_decision', '').lower().strip(),
            'question_meshes': entry.get('MESHES', []),
        })

    dist = Counter(q['label'] for q in fold_qs)
    print(f'\n  Loaded {len(fold_qs)} questions')
    for lbl, cnt in sorted(dist.items()):
        print(f'    {lbl:8s}: {cnt} ({100*cnt/len(fold_qs):.1f}%)')

    fold_results = load_ckpt(ckpt_path)
    prompt_preds = {pt: {} for pt in run_prompts}
    for pubid, r in fold_results.items():
        for pt in run_prompts:
            if pt in r.get('predictions', {}):
                prompt_preds[pt][pubid] = r['predictions'][pt]

    start = time.time()

    for q_idx, q in enumerate(tqdm(fold_qs, desc=f'Fold {fold_num}')):
        pubid = q['pubid']

        if pubid in fold_results:
            done = set(fold_results[pubid].get('predictions', {}).keys())
            todo = [pt for pt in run_prompts if pt not in done]
            if not todo:
                continue
        else:
            todo = run_prompts
            fold_results[pubid] = {
                'pubid': pubid, 'question': q['question'], 'label': q['label'],
                'predictions': {}, 'thinking_traces': {},
                'elhr_passages': None, 'query_mesh_terms': None, 'channel_stats': None,
            }

        if fold_results[pubid]['elhr_passages'] is None:
            passages, mesh_terms = elhr_kg_retrieve_with_source(q, k=K_PASSAGES)

            only_b = sum(1 for _,_,b in passages if     b['in_bm25'] and not b['in_mesh'] and not b['in_kg'])
            only_m = sum(1 for _,_,b in passages if not b['in_bm25'] and     b['in_mesh'] and not b['in_kg'])
            only_k = sum(1 for _,_,b in passages if not b['in_bm25'] and not b['in_mesh'] and     b['in_kg'])
            multi  = sum(1 for _,_,b in passages if sum([b['in_bm25'],b['in_mesh'],b['in_kg']]) > 1)
            n_cuis = passages[1][2]['n_kg_cuis'] if len(passages) > 1 else 0

            fold_results[pubid]['elhr_passages']    = [{'text': t, 'score': s, 'breakdown': bd} for t, s, bd in passages]
            fold_results[pubid]['query_mesh_terms'] = mesh_terms
            fold_results[pubid]['channel_stats']    = {
                'only_bm25': only_b, 'only_mesh': only_m, 'only_kg': only_k,
                'multi': multi, 'n_mesh_terms': len(mesh_terms), 'n_kg_cuis': n_cuis,
            }
        else:
            passages   = [(p['text'], p['score'], p['breakdown']) for p in fold_results[pubid]['elhr_passages']]
            mesh_terms = fold_results[pubid]['query_mesh_terms']

        for pt in todo:
            msgs = PROMPT_BUILDERS[pt](q['question'], passages, mesh_terms)
            content, thinking, tokens, key_id = api_manager.call(
                messages=msgs, temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS, question_id=f'{pubid}_{pt}')

            pred = parse_answer(content) if content else 'invalid'
            fold_results[pubid]['predictions'][pt]     = pred
            fold_results[pubid]['thinking_traces'][pt] = thinking or ''
            fold_results[pubid][f'raw_{pt}']           = content
            fold_results[pubid][f'key_{pt}']           = key_id
            fold_results[pubid][f'tokens_{pt}']        = tokens
            prompt_preds[pt][pubid] = pred

        if (q_idx + 1) % 5 == 0:
            save_ckpt(ckpt_path, fold_results)
            el       = time.time() - start
            done_now = sum(1 for r in fold_results.values() if run_prompts[0] in r.get('predictions', {}))
            remaining = len(fold_qs) - done_now
            eta = remaining / max(done_now / max(el, 0.01), 0.01)
            print(f'  [{done_now}/{len(fold_qs)}] {el:.0f}s | ETA {eta:.0f}s | tokens={api_manager.total_tokens:,}')

    save_ckpt(ckpt_path, fold_results)
    print(f'\n✓ Fold {fold_num} done in {time.time()-start:.1f}s')
    api_manager.status()

    label_map   = {q['pubid']: q['label'] for q in fold_qs}
    all_metrics = {}
    best_acc, best_pt = 0, None

    for pt in run_prompts:
        yt, yp = [], []
        for pubid, pred in prompt_preds[pt].items():
            if pubid in label_map:
                yt.append(label_map[pubid])
                yp.append(pred)
        if not yt:
            continue
        m = compute_metrics(yt, yp)
        all_metrics[pt] = m
        print_metrics(m, title=f'ELHR+KG | {MODEL_ID} | {pt.upper()} | Fold {fold_num} | k={K_PASSAGES}')
        if m['accuracy'] > best_acc:
            best_acc, best_pt = m['accuracy'], pt

    print(f'\n{"="*65}')
    print(f'CHANNEL ATTRIBUTION — Fold {fold_num}')
    print(f'{"="*65}')
    agg = Counter()
    for r in fold_results.values():
        cs = r.get('channel_stats', {})
        if cs:
            for kk, vv in cs.items(): agg[kk] += vv
    n_q = len(fold_results)
    print(f'  {n_q} questions × {K_PASSAGES} passages')
    print(f'  Only BM25 : {agg["only_bm25"]:4d}')
    print(f'  Only MeSH : {agg["only_mesh"]:4d}  ← vs 499A')
    print(f'  Only KG   : {agg["only_kg"]:4d}   ← new in KG-RAG')
    print(f'  Multiple  : {agg["multi"]:4d}')
    print(f'  Avg KG CUIs/q: {agg["n_kg_cuis"]/max(n_q,1):.1f}')

    out = {
        'experiment': {
            'fold'           : fold_num,
            'n_questions'    : len(fold_qs),
            'model'          : MODEL_ID,
            'retrieval'      : 'ELHR+KG: BM25+MeSH+KG (Dense disabled)',
            'weights'        : {'bm25': ALPHA_BM25, 'dense': ALPHA_DENSE,
                                'mesh': ALPHA_MESH,  'kg': ALPHA_KG},
            'kg_max_seed_cuis'  : KG_MAX_SEED_CUIS,
            'kg_allowed_rels'   : list(ALLOWED_KG_RELS),
            'k'              : K_PASSAGES,
            'temperature'    : TEMPERATURE,
            'mesh_active'    : MESH_AVAILABLE,
            'kg_active'      : KG_AVAILABLE,
            'dense_active'   : False,
            'timestamp'      : datetime.now().isoformat(),
            'total_tokens'   : api_manager.total_tokens,
        },
        'metrics'     : all_metrics,
        'best_prompt' : best_pt,
        'raw_results' : fold_results,
        'api_failures': api_manager.failed_requests,
    }
    with open(results_path, 'w') as f:
        json.dump(out, f, indent=2, ensure_ascii=False)
    print(f'\nResults saved → {results_path}')

    traces = {
        pubid: {'question': r['question'], 'label': r['label'],
                'predictions': r['predictions'],
                'thinking': r.get('thinking_traces', {}),
                'query_mesh_terms': r.get('query_mesh_terms', [])}
        for pubid, r in fold_results.items()
        if any(r.get('thinking_traces', {}).values())
    }
    with open(traces_path, 'w') as f:
        json.dump(traces, f, indent=2, ensure_ascii=False)
    print(f'Thinking traces → {traces_path} ({len(traces)} questions)')

    return fold_results, prompt_preds, all_metrics, best_pt


print('run_fold() defined ✓')

run_fold() defined ✓


## CELL 12: Clean Checkpoint (run before a fresh Fold 0 start)

In [18]:
#   Only run this cell to RESET fold 0. Comment out during normal runs.
for fpath in [CHECKPOINT_FILE, FINAL_RESULTS_FILE, THINKING_TRACES_FILE]:
    if fpath.exists():
        fpath.unlink()
        print(f'  Deleted: {fpath.name}')
    else:
        print(f'  Not found (clean): {fpath.name}')
print('\nClean. Safe to re-run Fold 0.')

  Not found (clean): fold0_checkpoint.json
  Not found (clean): fold0_final_results.json
  Not found (clean): fold0_thinking_traces.json

Clean. Safe to re-run Fold 0.


## CELL 13: Fold 0 — All 4 Prompt Types (finds best prompt)

In [19]:
fold0_results, fold0_preds, fold0_metrics, fold0_best_prompt = run_fold(
    fold_num     = 0,
    dev_path     = FOLD0_DEV,
    ckpt_path    = BASE_DIR / 'fold0_checkpoint.json',
    results_path = BASE_DIR / 'fold0_final_results.json',
    traces_path  = BASE_DIR / 'fold0_thinking_traces.json',
    prompt_types = PROMPT_TYPES_TO_TEST,
    best_prompt  = None,
)

summary = []
for pt, m in fold0_metrics.items():
    summary.append({'Prompt': pt, 'Acc%': m['accuracy'], 'F1%': m['f1_macro'],
                    'Kappa': m['kappa'], 'F1_maybe%': m.get('f_maybe', 0), 'Invalid': m['n_invalid']})
print('\n' + '='*65)
print('FOLD 0 SUMMARY')
print('='*65)
print(pd.DataFrame(summary).sort_values('Acc%', ascending=False).to_string(index=False))
print(f'\n Best prompt: {fold0_best_prompt}')

if fold0_best_prompt and fold0_best_prompt in fold0_metrics:
    m = fold0_metrics[fold0_best_prompt]
    print(f'\n{"="*65}')
    print('GO / NO-GO FOR FOLDS 2, 4, 6, 8')
    print(f'{"="*65}')
    print(f'  Accuracy  : {m["accuracy"]}%  (target ≥60%)')
    print(f'  F1 Macro  : {m["f1_macro"]}%')
    print(f'  Kappa     : {m["kappa"]}')
    if m['accuracy'] >= 60:
        print(f'   GO — proceed to folds 2,4,6,8 with prompt: {fold0_best_prompt}')
    elif m['accuracy'] >= 55:
        print(f'    MARGINAL — consider checking KG coverage before proceeding')
    else:
        print(f'   STOP — check KG loaded correctly and MeSH channel active')

FOLD 0 — ELHR+KG EVALUATION
  Prompts  : ['basic', 'cot', 'evidence', 'fewshot']
  Channels : BM25(α=0.45) | MeSH(γ=0.3) | KG(δ=0.25)
  KG active: True  |  MeSH active: True
  Model    : qwen/qwen3-32b  |  k=15

  Loaded 50 questions
    maybe   : 6 (12.0%)
    no      : 17 (34.0%)
    yes     : 27 (54.0%)
No checkpoint — fresh start


Fold 0:   0%|          | 0/50 [00:00<?, ?it/s]

  [5/50] 158s | ETA 1423s | tokens=64,615
  [10/50] 322s | ETA 1288s | tokens=126,081
  [15/50] 492s | ETA 1148s | tokens=187,585
  [20/50] 675s | ETA 1012s | tokens=249,849
  [25/50] 844s | ETA 844s | tokens=311,522
  [30/50] 1016s | ETA 677s | tokens=377,484
  [35/50] 1196s | ETA 513s | tokens=441,736
  [40/50] 1351s | ETA 338s | tokens=505,468
  [45/50] 1521s | ETA 169s | tokens=568,766
  [50/50] 1691s | ETA 0s | tokens=632,210

✓ Fold 0 done in 1690.9s

  Key  0: ok OK=  16 Fail= 0 Tokens=3,140
  Key  1: ok OK=  16 Fail= 0 Tokens=2,665
  Key  2: ok OK=  16 Fail= 0 Tokens=2,911
  Key  3: ok OK=  16 Fail= 0 Tokens=2,851
  Key  4: ok OK=  16 Fail= 0 Tokens=2,800
  Key  5: ok OK=  15 Fail= 0 Tokens=3,333
  Key  6: ok OK=  15 Fail= 0 Tokens=3,118
  Key  7: ok OK=  15 Fail= 0 Tokens=3,581
  Key  8: ok OK=  15 Fail= 0 Tokens=3,204
  Key  9: ok OK=  15 Fail= 0 Tokens=3,401
  Key 10: ok OK=  15 Fail= 0 Tokens=3,294
  Key 11: ok OK=  15 Fail= 0 Tokens=3,279
  Key 12: ok OK=  15 Fail= 0 Token

## CELL 14: Fold 2 — Best Prompt Only

In [21]:
FOLD_NUM    = 2
BEST_PROMPT = 'fewshot'   

fold2_results, fold2_preds, fold2_metrics, _ = run_fold(
    fold_num     = FOLD_NUM,
    dev_path     = INPUT_DIR / PQAL_DATASET_NAME / f'pqal_fold{FOLD_NUM}' / 'dev_set.json',
    ckpt_path    = BASE_DIR / f'fold{FOLD_NUM}_checkpoint.json',
    results_path = BASE_DIR / f'fold{FOLD_NUM}_final_results.json',
    traces_path  = BASE_DIR / f'fold{FOLD_NUM}_thinking_traces.json',
    prompt_types = [],
    best_prompt  = BEST_PROMPT,
)

FOLD 2 — ELHR+KG EVALUATION
  Prompts  : ['fewshot']
  Channels : BM25(α=0.45) | MeSH(γ=0.3) | KG(δ=0.25)
  KG active: True  |  MeSH active: True
  Model    : qwen/qwen3-32b  |  k=15

  Loaded 50 questions
    maybe   : 5 (10.0%)
    no      : 17 (34.0%)
    yes     : 28 (56.0%)
No checkpoint — fresh start


Fold 2:   0%|          | 0/50 [00:00<?, ?it/s]

  [5/50] 129s | ETA 1161s | tokens=647,455
  [10/50] 281s | ETA 1126s | tokens=661,819
  [15/50] 442s | ETA 1031s | tokens=677,193
  [20/50] 585s | ETA 878s | tokens=692,735
  [25/50] 732s | ETA 732s | tokens=707,890
  [30/50] 874s | ETA 583s | tokens=722,755
  [35/50] 1022s | ETA 438s | tokens=738,441
  [40/50] 1165s | ETA 291s | tokens=753,872
  [45/50] 1313s | ETA 146s | tokens=769,522
  [50/50] 1473s | ETA 0s | tokens=785,616

✓ Fold 2 done in 1472.7s

  Key  0: ok OK=  20 Fail= 0 Tokens=3,969
  Key  1: ok OK=  20 Fail= 0 Tokens=3,600
  Key  2: ok OK=  20 Fail= 0 Tokens=2,839
  Key  3: ok OK=  19 Fail= 0 Tokens=2,898
  Key  4: ok OK=  19 Fail= 0 Tokens=3,025
  Key  5: ok OK=  19 Fail= 0 Tokens=2,959
  Key  6: ok OK=  19 Fail= 0 Tokens=2,960
  Key  7: ok OK=  19 Fail= 0 Tokens=3,146
  Key  8: ok OK=  19 Fail= 0 Tokens=3,096
  Key  9: ok OK=  19 Fail= 0 Tokens=2,889
  Key 10: ok OK=  19 Fail= 0 Tokens=3,559
  Key 11: ok OK=  19 Fail= 0 Tokens=2,742
  Key 12: ok OK=  19 Fail= 0 Tokens

## CELL 15: Fold 4 — Best Prompt Only

In [22]:
FOLD_NUM    = 4
BEST_PROMPT = 'fewshot'

fold4_results, fold4_preds, fold4_metrics, _ = run_fold(
    fold_num     = FOLD_NUM,
    dev_path     = INPUT_DIR / PQAL_DATASET_NAME / f'pqal_fold{FOLD_NUM}' / 'dev_set.json',
    ckpt_path    = BASE_DIR / f'fold{FOLD_NUM}_checkpoint.json',
    results_path = BASE_DIR / f'fold{FOLD_NUM}_final_results.json',
    traces_path  = BASE_DIR / f'fold{FOLD_NUM}_thinking_traces.json',
    prompt_types = [],          # ignored when best_prompt is set
    best_prompt  = BEST_PROMPT,
)


FOLD 4 — ELHR+KG EVALUATION
  Prompts  : ['fewshot']
  Channels : BM25(α=0.45) | MeSH(γ=0.3) | KG(δ=0.25)
  KG active: True  |  MeSH active: True
  Model    : qwen/qwen3-32b  |  k=15

  Loaded 50 questions
    maybe   : 6 (12.0%)
    no      : 16 (32.0%)
    yes     : 28 (56.0%)
No checkpoint — fresh start


Fold 4:   0%|          | 0/50 [00:00<?, ?it/s]

  [5/50] 137s | ETA 1230s | tokens=801,037
  [10/50] 278s | ETA 1113s | tokens=815,890
  [15/50] 433s | ETA 1011s | tokens=831,336
  [20/50] 597s | ETA 895s | tokens=847,113
  [25/50] 741s | ETA 741s | tokens=862,782
  [30/50] 888s | ETA 592s | tokens=877,352
  [35/50] 1031s | ETA 442s | tokens=892,320
  [40/50] 1189s | ETA 297s | tokens=908,892
  [45/50] 1332s | ETA 148s | tokens=924,722
  [50/50] 1481s | ETA 0s | tokens=939,993

✓ Fold 4 done in 1481.5s

  Key  0: ok OK=  24 Fail= 0 Tokens=2,827
  Key  1: ok OK=  23 Fail= 0 Tokens=3,457
  Key  2: ok OK=  23 Fail= 0 Tokens=3,177
  Key  3: ok OK=  23 Fail= 0 Tokens=3,122
  Key  4: ok OK=  23 Fail= 0 Tokens=3,087
  Key  5: ok OK=  23 Fail= 0 Tokens=3,093
  Key  6: ok OK=  23 Fail= 0 Tokens=3,080
  Key  7: ok OK=  23 Fail= 0 Tokens=2,768
  Key  8: ok OK=  23 Fail= 0 Tokens=3,802
  Key  9: ok OK=  23 Fail= 0 Tokens=2,987
  Key 10: ok OK=  23 Fail= 0 Tokens=3,220
  Key 11: ok OK=  23 Fail= 0 Tokens=2,837
  Key 12: ok OK=  23 Fail= 0 Tokens

## CELL 16: Fold 6 — Best Prompt Only

In [23]:
FOLD_NUM    = 6
BEST_PROMPT = 'fewshot'

fold6_results, fold6_preds, fold6_metrics, _ = run_fold(
    fold_num     = FOLD_NUM,
    dev_path     = INPUT_DIR / PQAL_DATASET_NAME / f'pqal_fold{FOLD_NUM}' / 'dev_set.json',
    ckpt_path    = BASE_DIR / f'fold{FOLD_NUM}_checkpoint.json',
    results_path = BASE_DIR / f'fold{FOLD_NUM}_final_results.json',
    traces_path  = BASE_DIR / f'fold{FOLD_NUM}_thinking_traces.json',
    prompt_types = [],          # ignored when best_prompt is set
    best_prompt  = BEST_PROMPT,
)


FOLD 6 — ELHR+KG EVALUATION
  Prompts  : ['fewshot']
  Channels : BM25(α=0.45) | MeSH(γ=0.3) | KG(δ=0.25)
  KG active: True  |  MeSH active: True
  Model    : qwen/qwen3-32b  |  k=15

  Loaded 50 questions
    maybe   : 6 (12.0%)
    no      : 16 (32.0%)
    yes     : 28 (56.0%)
No checkpoint — fresh start


Fold 6:   0%|          | 0/50 [00:00<?, ?it/s]

  [5/50] 131s | ETA 1176s | tokens=955,141
  [10/50] 283s | ETA 1130s | tokens=971,438
  [15/50] 432s | ETA 1007s | tokens=987,165
  [20/50] 584s | ETA 876s | tokens=1,002,830
  [25/50] 739s | ETA 739s | tokens=1,019,384
  [30/50] 897s | ETA 598s | tokens=1,035,080
  [35/50] 1044s | ETA 447s | tokens=1,051,966
  [40/50] 1186s | ETA 297s | tokens=1,066,924
  [45/50] 1338s | ETA 149s | tokens=1,082,008
  [50/50] 1493s | ETA 0s | tokens=1,098,171

✓ Fold 6 done in 1492.8s

  Key  0: ok OK=  27 Fail= 0 Tokens=2,761
  Key  1: ok OK=  27 Fail= 0 Tokens=3,093
  Key  2: ok OK=  27 Fail= 0 Tokens=3,112
  Key  3: ok OK=  27 Fail= 0 Tokens=3,063
  Key  4: ok OK=  27 Fail= 0 Tokens=2,888
  Key  5: ok OK=  27 Fail= 0 Tokens=2,776
  Key  6: ok OK=  27 Fail= 0 Tokens=3,245
  Key  7: ok OK=  27 Fail= 0 Tokens=2,957
  Key  8: ok OK=  27 Fail= 0 Tokens=3,716
  Key  9: ok OK=  27 Fail= 0 Tokens=2,902
  Key 10: ok OK=  27 Fail= 0 Tokens=3,082
  Key 11: ok OK=  27 Fail= 0 Tokens=3,506
  Key 12: ok OK=  26 

## CELL 17: Fold 8 — Best Prompt Only

In [24]:
FOLD_NUM    = 8
BEST_PROMPT = 'fewshot'

fold8_results, fold8_preds, fold8_metrics, _ = run_fold(
    fold_num     = FOLD_NUM,
    dev_path     = INPUT_DIR / PQAL_DATASET_NAME / f'pqal_fold{FOLD_NUM}' / 'dev_set.json',
    ckpt_path    = BASE_DIR / f'fold{FOLD_NUM}_checkpoint.json',
    results_path = BASE_DIR / f'fold{FOLD_NUM}_final_results.json',
    traces_path  = BASE_DIR / f'fold{FOLD_NUM}_thinking_traces.json',
    prompt_types = [],          # ignored when best_prompt is set
    best_prompt  = BEST_PROMPT,
)


FOLD 8 — ELHR+KG EVALUATION
  Prompts  : ['fewshot']
  Channels : BM25(α=0.45) | MeSH(γ=0.3) | KG(δ=0.25)
  KG active: True  |  MeSH active: True
  Model    : qwen/qwen3-32b  |  k=15

  Loaded 50 questions
    maybe   : 6 (12.0%)
    no      : 17 (34.0%)
    yes     : 27 (54.0%)
No checkpoint — fresh start


Fold 8:   0%|          | 0/50 [00:00<?, ?it/s]

  [5/50] 164s | ETA 1479s | tokens=1,113,248
  [10/50] 329s | ETA 1315s | tokens=1,129,463
  [15/50] 494s | ETA 1152s | tokens=1,144,998
  [20/50] 640s | ETA 960s | tokens=1,160,628
  [25/50] 785s | ETA 785s | tokens=1,176,788
  [30/50] 933s | ETA 622s | tokens=1,191,874
  [35/50] 1091s | ETA 468s | tokens=1,206,696
  [40/50] 1254s | ETA 313s | tokens=1,221,709
  [45/50] 1398s | ETA 155s | tokens=1,238,367
  [50/50] 1536s | ETA 0s | tokens=1,255,680

✓ Fold 8 done in 1536.5s

  Key  0: ok OK=  31 Fail= 0 Tokens=2,958
  Key  1: ok OK=  31 Fail= 0 Tokens=3,664
  Key  2: ok OK=  31 Fail= 0 Tokens=3,363
  Key  3: ok OK=  31 Fail= 0 Tokens=3,684
  Key  4: ok OK=  31 Fail= 0 Tokens=2,989
  Key  5: ok OK=  31 Fail= 0 Tokens=3,787
  Key  6: ok OK=  31 Fail= 0 Tokens=3,682
  Key  7: ok OK=  31 Fail= 0 Tokens=3,565
  Key  8: ok OK=  31 Fail= 0 Tokens=3,154
  Key  9: ok OK=  31 Fail= 0 Tokens=3,125
  Key 10: ok OK=  30 Fail= 0 Tokens=3,105
  Key 11: ok OK=  30 Fail= 0 Tokens=3,161
  Key 12: ok OK

## CELL 18: CV Aggregation — Final 5-Fold Results
### Reads from saved JSON files — safe to run even if folds ran in separate sessions

In [25]:
fold_nums    = [0, 2, 4, 6, 8]
result_files = {f: BASE_DIR / f'fold{f}_final_results.json' for f in fold_nums}

rows, accs, f1s, kappas = [], [], [], []

print('='*65)
print('5-FOLD CV — ELHR+KG-RAG | BM25+MeSH+KG | k=15')
print(f'Model: {MODEL_ID}')
print('='*65)
print(f'  {"Fold":>6} | {"Acc%":>7} | {"F1%":>7} | {"Kappa":>7} | {"Valid":>7} | Prompt')
print(f'  {"-"*62}')

used_prompt = None
for fold in fold_nums:
    fpath = result_files[fold]
    if not fpath.exists():
        print(f'  Fold {fold:>2}  : NOT YET RUN')
        continue
    with open(fpath) as f:
        data = json.load(f)
    best_pt = data.get('best_prompt', 'fewshot')
    if best_pt not in data['metrics']:
        best_pt = list(data['metrics'].keys())[0]
    used_prompt = best_pt
    m = data['metrics'][best_pt]
    accs.append(m['accuracy'])
    f1s.append(m['f1_macro'])
    kappas.append(m['kappa'])
    rows.append(m)
    print(f'  Fold {fold:>2}  | {m["accuracy"]:>7.2f} | {m["f1_macro"]:>7.2f} | {m["kappa"]:>7.4f} | {m["n_valid"]:>2}/{m["n_total"]:>2} | {best_pt}')

if accs:
    print(f'  {"-"*62}')
    print(f'  {"Mean":>6}  | {np.mean(accs):>7.2f} | {np.mean(f1s):>7.2f} | {np.mean(kappas):>7.4f}')
    print(f'  {"±Std":>6}  | {np.std(accs):>7.2f} | {np.std(f1s):>7.2f} | {np.std(kappas):>7.4f}')

    print(f'\n{"="*65}')
    print(f'  COMPARISON TABLE')
    print(f'{"="*65}')
    print(f'  {"System":<40} | {"Acc%":>7} | {"F1%":>7} | {"Kappa":>7}')
    print(f'  {"-"*65}')
    print(f'  {"Random baseline (3-class)":<40} | {"33.3":>7} | {"33.3":>7} | {"0.000":>7}')
    print(f'  {"Jin et al. 2019 SVM (PubMedQA paper)":<40} | {"55.8":>7} | {"—":>7} | {"—":>7}')
    print(f'  {"Jin et al. 2019 BioBERT fine-tuned":<40} | {"68.1":>7} | {"—":>7} | {"—":>7}')
    print(f'  {"499A CV baseline (BM25+Dense)":<40} | {"52.8":>7} | {"34.1":>7} | {"0.074":>7}')
    print(f'  {"ELHR (BM25+MeSH, no KG)":<40} | {"—":>7} | {"—":>7} | {"—":>7}')
    print(f'  {"ELHR+KG — this work":<40} | {np.mean(accs):>7.2f} | {np.mean(f1s):>7.2f} | {np.mean(kappas):>7.4f}')
    print(f'  {"  vs 499A":<40} | {np.mean(accs)-52.8:>+7.2f} | {np.mean(f1s)-34.1:>+7.2f} | {np.mean(kappas)-0.074:>+7.4f}')
    print(f'  {"  vs Jin SVM":<40} | {np.mean(accs)-55.8:>+7.2f} | {"—":>7} | {"—":>7}')

    print(f'\n{"="*65}')
    print(f'KG CHANNEL CONTRIBUTION — all folds combined')
    print(f'{"="*65}')
    total_bm25, total_mesh, total_kg, total_multi = 0, 0, 0, 0
    for fold in fold_nums:
        fpath = result_files[fold]
        if not fpath.exists(): continue
        with open(fpath) as f:
            data = json.load(f)
        for r in data['raw_results'].values():
            cs = r.get('channel_stats', {})
            if cs:
                total_bm25  += cs.get('only_bm25', 0)
                total_mesh  += cs.get('only_mesh', 0)
                total_kg    += cs.get('only_kg',   0)
                total_multi += cs.get('multi', 0)

    grand_total = total_bm25 + total_mesh + total_kg + total_multi
    if grand_total > 0:
        print(f'  Only BM25  : {total_bm25:>6,}  ({100*total_bm25/grand_total:.1f}%)')
        print(f'  Only MeSH  : {total_mesh:>6,}  ({100*total_mesh/grand_total:.1f}%)  ← new vs 499A')
        print(f'  Only KG    : {total_kg:>6,}  ({100*total_kg/grand_total:.1f}%)   ← KG-RAG contribution')
        print(f'  Multiple   : {total_multi:>6,}  ({100*total_multi/grand_total:.1f}%)')

    cv_summary = {
        'system'  : f'ELHR+KG: BM25+MeSH+KG | {used_prompt} | {MODEL_ID} | k={K_PASSAGES}',
        'weights' : {'bm25': ALPHA_BM25, 'mesh': ALPHA_MESH, 'kg': ALPHA_KG},
        'kg_max_seed_cuis': KG_MAX_SEED_CUIS,
        'folds'   : fold_nums,
        'mean_accuracy': round(np.mean(accs), 2), 'std_accuracy': round(np.std(accs), 2),
        'mean_f1'      : round(np.mean(f1s),  2), 'std_f1'      : round(np.std(f1s), 2),
        'mean_kappa'   : round(np.mean(kappas), 4), 'std_kappa'  : round(np.std(kappas), 4),
        'baseline_acc' : 52.8, 'baseline_f1': 34.1, 'baseline_kappa': 0.074,
        'improvement_acc'  : round(np.mean(accs) - 52.8, 2),
        'improvement_f1'   : round(np.mean(f1s)  - 34.1, 2),
        'improvement_kappa': round(np.mean(kappas) - 0.074, 4),
        'kg_only_passages'  : total_kg,
        'mesh_only_passages': total_mesh,
        'bm25_only_passages': total_bm25,
    }
    with open(BASE_DIR / 'elhr_kg_cv_summary.json', 'w') as f:
        json.dump(cv_summary, f, indent=2)
    print(f'\nCV summary saved → elhr_kg_cv_summary.json')

5-FOLD CV — ELHR+KG-RAG | BM25+MeSH+KG | k=15
Model: qwen/qwen3-32b
    Fold |    Acc% |     F1% |   Kappa |   Valid | Prompt
  --------------------------------------------------------------
  Fold  0  |   78.00 |   63.74 |  0.6113 | 50/50 | basic
  Fold  2  |   82.00 |   66.13 |  0.6788 | 50/50 | fewshot
  Fold  4  |   76.00 |   65.97 |  0.5751 | 50/50 | fewshot
  Fold  6  |   76.00 |   64.99 |  0.5781 | 50/50 | fewshot
  Fold  8  |   72.00 |   62.55 |  0.4946 | 50/50 | fewshot
  --------------------------------------------------------------
    Mean  |   76.80 |   64.68 |  0.5876
    ±Std  |    3.25 |    1.36 |  0.0596

  COMPARISON TABLE
  System                                   |    Acc% |     F1% |   Kappa
  -----------------------------------------------------------------
  Random baseline (3-class)                |    33.3 |    33.3 |   0.000
  Jin et al. 2019 SVM (PubMedQA paper)     |    55.8 |       — |       —
  Jin et al. 2019 BioBERT fine-tuned       |    68.1 |       — |